# Notebook Version Marker

This is `lt_formal_finetuning_workflow_cpu_v4.ipynb`.

This version is optimized for CPU-only training as much as possible while still using LoRA.
If you do not see this exact text, you opened the wrong notebook.


# Lithuanian Informal-to-Formal Fine-Tuning Workflow

This notebook is self-contained and designed for weak hardware.

What changed in this CPU-friendly version:

- smaller multilingual model,
- LoRA is still used,
- very light default settings,
- optional quick subset mode,
- visible training progress bar,
- no `trl` dependency.


## 1. Install dependencies

In [ ]:
%pip install torch transformers datasets peft accelerate sentencepiece safetensors

## 2. Imports and paths

In [ ]:
import json
import os
import random
import re
import statistics
from difflib import SequenceMatcher
from pathlib import Path

os.environ['PYTHONUTF8'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'

import torch
from datasets import Dataset
from peft import AutoPeftModelForSeq2SeqLM, LoraConfig, TaskType, get_peft_model
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

project_dir = Path.cwd()
dataset_path = project_dir / 'lt_formal_dataset_303.json'
prepared_dir = project_dir / 'prepared'
outputs_dir = project_dir / 'outputs'
model_output_dir = outputs_dir / 'mt5_small_lora_cpu'
prepared_dir.mkdir(exist_ok=True)
outputs_dir.mkdir(exist_ok=True)

print('Project directory:', project_dir)
print('Dataset exists:', dataset_path.exists())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))
else:
    print('CPU-only mode detected.')


## 3. Helper functions

In [ ]:
PROMPT_PREFIX = 'Rewrite this informal Lithuanian text into formal, grammatically correct Lithuanian:'


def load_rows(path: Path) -> list[dict]:
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path: Path, payload: object) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')


def normalize_text(text: str) -> str:
    text = text.casefold().strip()
    return re.sub(r'\s+', ' ', text)


def evaluate_predictions(reference_rows: list[dict], predictions: list[dict]) -> dict:
    prediction_map = {int(row['id']): row['prediction'] for row in predictions}
    scored_rows = []
    exact_match = 0
    normalized_exact_match = 0
    char_similarity_total = 0.0

    for row in reference_rows:
        predicted = prediction_map.get(int(row['id']), '')
        gold = row['formal_text']
        is_exact = predicted == gold
        is_normalized_exact = normalize_text(predicted) == normalize_text(gold)
        similarity = SequenceMatcher(None, normalize_text(predicted), normalize_text(gold)).ratio()

        exact_match += int(is_exact)
        normalized_exact_match += int(is_normalized_exact)
        char_similarity_total += similarity

        scored_rows.append({
            'id': row['id'],
            'informal_text': row['informal_text'],
            'gold': gold,
            'prediction': predicted,
            'exact_match': is_exact,
            'normalized_exact_match': is_normalized_exact,
            'char_similarity': round(similarity, 4),
        })

    total = len(reference_rows)
    return {
        'total_examples': total,
        'exact_match': round(exact_match / total, 4),
        'normalized_exact_match': round(normalized_exact_match / total, 4),
        'average_char_similarity': round(char_similarity_total / total, 4),
        'details': scored_rows,
    }


class NotebookProgressCallback(TrainerCallback):
    def __init__(self):
        self.progress_bar = None
        self.last_epoch_printed = None

    def on_train_begin(self, args, state, control, **kwargs):
        total_steps = state.max_steps if state.max_steps is not None else 0
        print(f'Training started. Total epochs: {args.num_train_epochs}, total optimization steps: {total_steps}')
        self.progress_bar = tqdm(total=total_steps, desc='Training', leave=True)

    def on_step_end(self, args, state, control, **kwargs):
        if self.progress_bar is not None:
            self.progress_bar.n = state.global_step
            epoch_value = 0.0 if state.epoch is None else state.epoch
            self.progress_bar.set_postfix(epoch=f'{epoch_value:.2f}')
            self.progress_bar.refresh()

    def on_epoch_begin(self, args, state, control, **kwargs):
        epoch_value = 0.0 if state.epoch is None else state.epoch + 1
        epoch_number = int(epoch_value)
        if self.last_epoch_printed != epoch_number:
            print(f'Starting epoch {epoch_number}/{int(args.num_train_epochs)}')
            self.last_epoch_printed = epoch_number

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            cleaned = {k: round(v, 4) if isinstance(v, float) else v for k, v in logs.items()}
            print(cleaned)

    def on_train_end(self, args, state, control, **kwargs):
        if self.progress_bar is not None:
            self.progress_bar.n = state.global_step
            self.progress_bar.refresh()
            self.progress_bar.close()
        print('Training finished.')


## 4. Load dataset and inspect it

In [ ]:
rows = load_rows(dataset_path)
train_rows = [row for row in rows if row['split'] == 'train']
test_rows = [row for row in rows if row['split'] == 'test']

informal_lengths = [len(row['informal_text']) for row in rows]
formal_lengths = [len(row['formal_text']) for row in rows]

stats = {
    'total_rows': len(rows),
    'train_rows': len(train_rows),
    'test_rows': len(test_rows),
    'avg_informal_chars': round(statistics.mean(informal_lengths), 2),
    'avg_formal_chars': round(statistics.mean(formal_lengths), 2),
    'median_informal_chars': statistics.median(informal_lengths),
    'median_formal_chars': statistics.median(formal_lengths),
}
stats

In [ ]:
rows[:3]

## 5. CPU-friendly training configuration

In [ ]:
BASE_MODEL = 'google/mt5-small'
NUM_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
MAX_INPUT_LENGTH = 96
MAX_TARGET_LENGTH = 96
USE_QUICK_CPU_MODE = True
MAX_TRAIN_SAMPLES = 96 if USE_QUICK_CPU_MODE else None
MAX_TEST_SAMPLES = 24 if USE_QUICK_CPU_MODE else None

print('Base model:', BASE_MODEL)
print('Quick CPU mode:', USE_QUICK_CPU_MODE)
print('Max train samples:', MAX_TRAIN_SAMPLES)
print('Max test samples:', MAX_TEST_SAMPLES)


## 6. Prepare training subsets

In [ ]:
working_train_rows = train_rows[:MAX_TRAIN_SAMPLES] if MAX_TRAIN_SAMPLES is not None else train_rows
working_test_rows = test_rows[:MAX_TEST_SAMPLES] if MAX_TEST_SAMPLES is not None else test_rows

print('Working train rows:', len(working_train_rows))
print('Working test rows:', len(working_test_rows))


## 7. Hardware note

In [ ]:
device_type = 'GPU' if torch.cuda.is_available() else 'CPU'
print('Training device:', device_type)
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    print('This notebook is configured for CPU-friendly training, but it may still take some time.')


## 8. Fine-tune with LoRA

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=['q', 'v'],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


def preprocess_example(example: dict) -> dict:
    source_text = f"{PROMPT_PREFIX}\n\n{example['informal_text']}"
    model_inputs = tokenizer(
        source_text,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding='max_length',
    )
    labels = tokenizer(
        text_target=example['formal_text'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding='max_length',
    )
    label_ids = labels['input_ids']
    label_ids = [token_id if token_id != tokenizer.pad_token_id else -100 for token_id in label_ids]
    model_inputs['labels'] = label_ids
    return model_inputs


train_dataset = Dataset.from_list(working_train_rows)
eval_dataset = Dataset.from_list(working_test_rows)
train_tokenized = train_dataset.map(preprocess_example, remove_columns=train_dataset.column_names)
eval_tokenized = eval_dataset.map(preprocess_example, remove_columns=eval_dataset.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = TrainingArguments(
    output_dir=str(model_output_dir),
    num_train_epochs=NUM_EPOCHS,
    learning_rate=5e-4,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    logging_steps=1,
    save_strategy='no',
    eval_strategy='no',
    per_device_eval_batch_size=1,
    report_to='none',
    remove_unused_columns=False,
    disable_tqdm=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
    callbacks=[NotebookProgressCallback()],
)

trainer.train()
trainer.save_model(str(model_output_dir))
tokenizer.save_pretrained(str(model_output_dir))

print('Saved model to:', model_output_dir)


## 9. Generate predictions

In [ ]:
prediction_tokenizer = AutoTokenizer.from_pretrained(model_output_dir, use_fast=True)
prediction_model = AutoPeftModelForSeq2SeqLM.from_pretrained(model_output_dir)
prediction_model.eval()

test_predictions = []
for row in working_test_rows:
    prompt = f"{PROMPT_PREFIX}\n\n{row['informal_text']}"
    encoded = prediction_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_INPUT_LENGTH)
    with torch.no_grad():
        output_ids = prediction_model.generate(
            **encoded,
            max_new_tokens=MAX_TARGET_LENGTH,
            do_sample=False,
        )
    prediction = prediction_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    test_predictions.append({'id': row['id'], 'prediction': prediction})

write_json(outputs_dir / 'test_predictions_cpu_v4.json', test_predictions)
print('Saved predictions:', outputs_dir / 'test_predictions_cpu_v4.json')
len(test_predictions)


## 10. Evaluate predictions

In [ ]:
evaluation_report = evaluate_predictions(working_test_rows, test_predictions)
write_json(outputs_dir / 'evaluation_report_cpu_v4.json', evaluation_report)
{k: v for k, v in evaluation_report.items() if k != 'details'}


In [ ]:
evaluation_report['details'][:5]

## 11. Test on a new input

In [ ]:
demo_text = 'nu ka cia dabar padarei'

In [ ]:
demo_prompt = f"{PROMPT_PREFIX}\n\n{demo_text}"
demo_encoded = prediction_tokenizer(demo_prompt, return_tensors='pt', truncation=True, max_length=MAX_INPUT_LENGTH)
with torch.no_grad():
    demo_output_ids = prediction_model.generate(
        **demo_encoded,
        max_new_tokens=MAX_TARGET_LENGTH,
        do_sample=False,
    )
demo_prediction = prediction_tokenizer.decode(demo_output_ids[0], skip_special_tokens=True).strip()
print('Input:', demo_text)
print('Output:', demo_prediction)


## 12. Notes

This `v4` notebook is intentionally lightweight.

If you want a stronger final experiment later, you can:

- set `USE_QUICK_CPU_MODE = False`,
- increase `NUM_EPOCHS` to `2` or `3`,
- or move to a GPU machine.

For now, this version is meant to actually finish on CPU hardware.
